# Module 2 — Drug-likeness Analysis (Lipinski's Rule of Five)

**Run notebook 01 first** — this notebook reads `data/compounds.csv` which that notebook creates.

Lipinski's Rule of Five checks whether a compound is likely to be absorbed orally as a drug:

| Property | Must be |
|---|---|
| Molecular Weight (MW) | ≤ 500 Da |
| LogP (lipophilicity) | ≤ 5 |
| H-bond Donors (HBD) | ≤ 5 |
| H-bond Acceptors (HBA) | ≤ 10 |

We compute these using **RDKit**, the gold-standard cheminformatics library.

In [ ]:
# ── CELL 1: Install RDKit (takes ~1 minute in Colab) ──
!pip install rdkit-pypi --quiet
print("✓ RDKit installed")

In [ ]:
# ── CELL 2: Upload compounds.csv ──
# If you ran notebook 01 in a different Colab session,
# the file won't be here. Upload it manually.
import os

if not os.path.exists("data/compounds.csv"):
    print("compounds.csv not found. Uploading now...")
    from google.colab import files
    os.makedirs("data", exist_ok=True)
    uploaded = files.upload()  # upload your compounds.csv here
    for fname in uploaded:
        os.rename(fname, f"data/{fname}")
    print("✓ File uploaded")
else:
    print("✓ compounds.csv found")

os.makedirs("figures", exist_ok=True)

In [ ]:
# ── CELL 3: Import libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # suppress RDKit warnings

print("✓ Libraries imported")

In [ ]:
# ── CELL 4: Load dataset ──
df = pd.read_csv("data/compounds.csv")
df_valid = df[df["canonical_smiles"].notna()].copy()

print(f"Total compounds: {len(df)}")
print(f"Compounds with SMILES (usable by RDKit): {len(df_valid)}")
df_valid[["compound_name", "canonical_smiles", "bioactivity"]].head()

In [ ]:
# ── CELL 5: Compute Lipinski descriptors ──
def compute_lipinski(smiles):
    """
    Parse SMILES with RDKit and compute four Lipinski descriptors.
    MW   = Molecular Weight
    LogP = Lipophilicity (Wildman-Crippen)
    HBD  = Hydrogen Bond Donors
    HBA  = Hydrogen Bond Acceptors
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"MW": None, "LogP": None, "HBD": None, "HBA": None}
    return {
        "MW":   round(Descriptors.ExactMolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "HBD":  rdMolDescriptors.CalcNumHBD(mol),
        "HBA":  rdMolDescriptors.CalcNumHBA(mol)
    }

lipinski_data  = df_valid["canonical_smiles"].apply(compute_lipinski)
df_lipinski    = pd.DataFrame(lipinski_data.tolist())
df_valid       = pd.concat([df_valid.reset_index(drop=True), df_lipinski], axis=1)

print("✓ Descriptors computed")
df_valid[["compound_name", "MW", "LogP", "HBD", "HBA"]]

In [ ]:
# ── CELL 6: Apply Lipinski filter ──
def lipinski_pass(row):
    try:
        return (
            row["MW"]   <= 500 and
            row["LogP"] <= 5   and
            row["HBD"]  <= 5   and
            row["HBA"]  <= 10
        )
    except:
        return False

df_valid["lipinski_pass"] = df_valid.apply(lipinski_pass, axis=1)

passed = df_valid["lipinski_pass"].sum()
total  = len(df_valid)
print(f"Compounds PASSING Lipinski Ro5: {passed} / {total}")
print(f"Compounds FAILING Lipinski Ro5: {total - passed} / {total}")
print()
print(df_valid[["compound_name", "MW", "LogP", "HBD", "HBA", "lipinski_pass"]].to_string(index=False))

In [ ]:
# ── CELL 7: Scatter plot — MW vs LogP ──
fig, ax = plt.subplots(figsize=(11, 6))

colors = ["#1565C0" if p else "#C62828" for p in df_valid["lipinski_pass"]]

ax.scatter(
    df_valid["MW"], df_valid["LogP"],
    c=colors, s=110, edgecolors="white", linewidths=0.8, zorder=3
)

for _, row in df_valid.iterrows():
    if pd.notna(row["MW"]) and pd.notna(row["LogP"]):
        ax.annotate(
            row["compound_name"],
            (row["MW"], row["LogP"]),
            textcoords="offset points", xytext=(6, 5),
            fontsize=8, color="#333"
        )

ax.axvline(x=500, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.axhline(y=5,   color="gray", linestyle=":",  linewidth=1, alpha=0.6)
ax.axvspan(0, 500, alpha=0.04, color="green")
ax.axhspan(-10, 5, alpha=0.04, color="green")

ax.text(505, ax.get_ylim()[0] + 0.2, "MW = 500", fontsize=8, color="gray")
ax.text(ax.get_xlim()[0] + 5, 5.1,  "LogP = 5", fontsize=8, color="gray")

pass_patch = mpatches.Patch(color="#1565C0", label="Passes Lipinski Ro5")
fail_patch = mpatches.Patch(color="#C62828", label="Fails Lipinski Ro5")
ax.legend(handles=[pass_patch, fail_patch], fontsize=9)

ax.set_xlabel("Molecular Weight (Da)", fontsize=11)
ax.set_ylabel("LogP (lipophilicity)", fontsize=11)
ax.set_title("Lipinski Drug-likeness: Endophyte Bioactive Compounds",
             fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/lipinski_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved to figures/lipinski_scatter.png")

In [ ]:
# ── CELL 8: Heatmap of all descriptors ──
df_heat = df_valid[["compound_name", "MW", "LogP", "HBD", "HBA"]].dropna()
df_heat = df_heat.set_index("compound_name")
df_norm = (df_heat - df_heat.min()) / (df_heat.max() - df_heat.min())

fig, ax = plt.subplots(figsize=(7, len(df_norm) * 0.48 + 2))
sns.heatmap(
    df_norm, annot=df_heat, fmt=".1f",
    cmap="YlOrRd", linewidths=0.5, ax=ax,
    cbar_kws={"label": "Normalised value"}
)
ax.set_title("Lipinski Descriptors — Endophyte Compounds",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Descriptor")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("figures/lipinski_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved to figures/lipinski_heatmap.png")

In [ ]:
# ── CELL 9: Download figures ──
from google.colab import files
files.download("figures/lipinski_scatter.png")
files.download("figures/lipinski_heatmap.png")
print("✓ Downloaded! Upload both images to your GitHub repo under figures/")

## Discussion

**Why does Taxol fail Lipinski?** Taxol (MW ~854 Da) violates the molecular weight rule. This is expected — Lipinski's rules were designed for small-molecule oral drugs. Taxol is administered intravenously and uses special membrane transport mechanisms. Natural products frequently violate one or more Lipinski rules, which is why the "rule of five" is a guideline, not a law. This nuance is an important discussion point in computational drug discovery.